# 01 - Visual Exploration of Spain CPI

Initial visual inspection of the general CPI index and 13 ECOICOP subgroups (monthly, 2002–2024).

**Input:** `data/processed/ipc_spain_index.parquet`  
**Establishes:** Series range, trend, seasonal pattern, train/val/test splits

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path('..').resolve()         # tfg-forecasting/
MONOREPO = ROOT.parent              # tfg-ipc-mcp/
sys.path.insert(0, str(MONOREPO))

from shared.constants import DATE_TRAIN_END, DATE_VAL_END, FREQ

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

## 1. Data Loading

In [ ]:
DATA_PATH = ROOT / "data" / "processed" / "ipc_spain_index.parquet"
df = pd.read_parquet(DATA_PATH)
print(f"Range: {df.index.min().date()} -> {df.index.max().date()}")
print(f"Shape: {df.shape}")
df.head()

In [3]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
indice_general,289.0,78.193076,10.498942,58.717,71.587,78.852,82.170,101.289
01_alimentos_bebidas,289.0,69.912637,12.972287,49.779,63.252,68.081,73.499,101.797
02_bebidas_alcoholicas_tabaco,289.0,69.556478,18.490770,38.396,50.307,77.208,82.281,103.245
03_vestido_calzado,289.0,90.855789,7.472154,76.673,84.027,92.140,96.066,108.042
04_vivienda_agua_electricidad,289.0,77.316291,13.689051,51.962,67.379,80.712,84.072,110.654
05_muebles_hogar,289.0,85.833471,7.139773,71.792,81.944,86.457,87.496,100.585
06_sanidad,289.0,88.872474,5.454803,79.900,83.788,90.636,92.445,101.052
07_transporte,289.0,79.793415,12.347033,56.407,70.354,81.252,85.745,105.418
08_informacion_comunicaciones,289.0,125.234765,32.926989,96.543,101.881,107.579,136.829,229.772
09_ocio_cultura,289.0,84.229270,7.637374,68.671,78.852,85.098,87.927,104.939


In [ ]:
# Verify frequency and gaps
expected = pd.date_range(df.index.min(), df.index.max(), freq="MS")
missing = expected.difference(df.index)
print(f"Expected months: {len(expected)}")
print(f"Present months: {len(df)}")
print(f"Gaps: {len(missing)}")
if len(missing) > 0:
    print("Missing dates:", missing.tolist())

## 2. Full Series - General Index

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df["indice_general"], linewidth=1.2, color="#1f77b4")

# Train/val/test split lines
for label, date, color in [
    ("Train end", DATE_TRAIN_END, "green"),
    ("Val end", DATE_VAL_END, "orange"),
]:
    ax.axvline(pd.Timestamp(date), color=color, linestyle="--", alpha=0.7, label=label)

ax.set_title("Spain CPI - General Index (level, base 2024)")
ax.set_ylabel("Index")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 3. Year-on-Year Change (YoY %)

In [ ]:
yoy = df["indice_general"].pct_change(12) * 100

fig, ax = plt.subplots()
ax.plot(yoy.index, yoy, linewidth=1, color="#d62728")
ax.axhline(0, color="black", linewidth=0.5)
ax.axhline(2, color="grey", linewidth=0.5, linestyle=":", label="ECB Target (2%)")

ax.set_title("Spain CPI - Year-on-Year Change (%)")
ax.set_ylabel("%")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 4. All ECOICOP Subgroups

In [ ]:
subgroups = [c for c in df.columns if c != "indice_general"]

fig, ax = plt.subplots(figsize=(16, 7))
for col in subgroups:
    ax.plot(df.index, df[col], linewidth=0.8, alpha=0.7, label=col)

ax.set_title("ECOICOP Subgroups - Index Level")
ax.set_ylabel("Index")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

## 5. Monthly Distribution (box plot)

In [ ]:
monthly_change = df["indice_general"].pct_change() * 100
monthly_df = pd.DataFrame({"month": df.index.month, "change_pct": monthly_change})

fig, ax = plt.subplots(figsize=(12, 5))
monthly_df.boxplot(column="change_pct", by="month", ax=ax)
ax.set_title("Monthly Change Distribution (%) by month")
ax.set_xlabel("Month")
ax.set_ylabel("Monthly Change (%)")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 6. Train / Validation / Test Splits

In [ ]:
train = df.loc[:DATE_TRAIN_END]
val   = df.loc[DATE_TRAIN_END:DATE_VAL_END].iloc[1:]
test  = df.loc[DATE_VAL_END:].iloc[1:]

fig, ax = plt.subplots()
ax.plot(train.index, train["indice_general"], label=f"Train ({len(train)})", color="#2ca02c")
ax.plot(val.index, val["indice_general"], label=f"Val ({len(val)})", color="#ff7f0e")
ax.plot(test.index, test["indice_general"], label=f"Test ({len(test)})", color="#d62728")

ax.set_title("Spain CPI - Temporal Splits")
ax.set_ylabel("Index")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

print(f"Train: {train.index.min().date()} -> {train.index.max().date()} ({len(train)} months)")
print(f"Val:   {val.index.min().date()} -> {val.index.max().date()} ({len(val)} months)")
print(f"Test:  {test.index.min().date()} -> {test.index.max().date()} ({len(test)} months)")

## 7. Summary

In [ ]:
print("="*60)
print("SPAIN CPI - EXPLORATION SUMMARY")
print("="*60)
print(f"Temporal range:    {df.index.min().date()} -> {df.index.max().date()}")
print(f"Total months:      {len(df)}")
print(f"Frequency:         Monthly (MS)")
print(f"Gaps:              {len(missing)}")
print(f"NaN (general index): {df['indice_general'].isna().sum()}")
print(f"Subgroups:         {len(subgroups)}")
print(f"Min index:         {df['indice_general'].min():.3f} ({df['indice_general'].idxmin().date()})")
print(f"Max index:         {df['indice_general'].max():.3f} ({df['indice_general'].idxmax().date()})")
print("="*60)